# Stocks B3 — do COTAHIST bruto às features

Ponto de partida: o parquet gerado por `data_generator/generate.py stocks_b3` (séries históricas
COTAHIST da B3, um arquivo por ano em `datasets/stocks_b3`). Objetivo: uma tabela de features por
ação e por pregão, pronta para alimentar um modelo.

O notebook tem dois blocos:

1. **Setup e dataset otimizado** — importações, sessão Spark, leitura do parquet e conversão para
   Delta particionado por ano.
2. **Tratamento e features** — filtro de ações, normalização de preço, log e retorno, z-score
   modificado, estocástico lento, média móvel de 90 dias, array dos últimos 6 meses e volume.

Cada passo é uma célula de texto dizendo **o que fazer**, **por quê**, **quais colunas entram e
saem** e **quais funções do PySpark resolvem** — seguida de uma célula de código vazia para você
implementar.

Referência rápida das colunas do parquet (nomes vêm do layout oficial da B3):

| Coluna | Tipo | Conteúdo |
|---|---|---|
| `trade_date` | date | Data do pregão |
| `bdi_code` | string | Código BDI (tipo de papel: `02` = lote padrão) |
| `ticker` | string | Código de negociação (`PETR4`) |
| `market_type` | string | Tipo de mercado (`010` = à vista, `070`/`080` = opções) |
| `company_name`, `specification` | string | Nome resumido e espécie (`ON`, `PN`) |
| `open_price`, `high_price`, `low_price`, `close_price` | double | Preços do dia, em R$ |
| `average_price`, `best_bid_price`, `best_ask_price` | double | Preço médio, melhor compra e venda |
| `total_trades` | long | Número de negócios |
| `total_quantity` | long | Quantidade de títulos negociados |
| `total_volume` | double | Volume financeiro em R$ |
| `quote_factor` | long | Fator de cotação (1 = por ação, 1000 = por lote de mil) |
| `strike_price`, `expiration_date`, `forward_term`, ... | — | Campos de opção/termo (saem no filtro) |
| `isin_code` | string | ISIN, estável mesmo quando o ticker muda |

## Bloco 1 — Setup e dataset otimizado

### Passo 1 — Importações

Importe:

- `Path` de `pathlib` — para resolver os diretórios de dados sem caminho absoluto.
- `configure_spark_with_delta_pip` de `delta` e `DeltaTable` de `delta.tables` — o primeiro injeta
  o jar do Delta na sessão; o segundo tem `isDeltaTable`, usado no passo 4 para pular a conversão
  se já foi feita.
- `SparkSession` e `Window` de `pyspark.sql`, e `functions as F`.
- `InteractiveShell` de `IPython.core.interactiveshell` e defina
  `InteractiveShell.ast_node_interactivity = "all"` para que toda expressão solta da célula seja
  exibida, não só a última (útil para `df.count()` no meio da célula).

`Window` é a peça central do bloco 2: todas as features são janelas por `ticker` ordenadas por
`trade_date`.

In [ ]:
from pathlib import Path

from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from IPython.core.interactiveshell import InteractiveShell
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

InteractiveShell.ast_node_interactivity = "all"

### Passo 2 — Sessão Spark

Defina três caminhos a partir do diretório `datasets/` (suba pelos `Path.cwd().parents` até achar
um que contenha `datasets/`, como no notebook `xgboost_movies_imdb`):

- `SOURCE = datasets/stocks_b3` — parquet bruto.
- `TARGET = datasets/stocks_b3_delta` — tabela Delta otimizada (saída do bloco 1).
- `FEATURES = datasets/stocks_b3_features` — tabela de features (saída do bloco 2).

Crie a sessão com `configure_spark_with_delta_pip(SparkSession.builder...).getOrCreate()` e estas
configurações — entenda cada uma antes de copiar:

- `.master("local[4]")` — 4 threads locais; ajuste ao número de núcleos.
- `spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension` e
  `spark.sql.catalog.spark_catalog = org.apache.spark.sql.delta.catalog.DeltaCatalog` — ligam o
  Delta Lake.
- `spark.driver.memory = 6g` — em modo local o driver é também o executor; o COTAHIST completo
  (1986–hoje) tem ~100 M de linhas.
- `spark.sql.parquet.compression.codec = zstd` — arquivos menores que o snappy padrão, um pouco mais
  de CPU na escrita.
- `spark.databricks.delta.optimizeWrite.enabled = true` e `autoCompact.enabled = true` — o Delta
  agrupa arquivos pequenos na escrita e compacta depois; evita o problema de "muitos arquivos
  pequenos" das partições por ano.
- `spark.databricks.delta.optimize.maxFileSize = 268435456` (256 MiB) — teto do arquivo que a
  compactação produz. Com o padrão (1 GiB) um ano inteiro vira um arquivo só e a leitura desse ano
  cai para uma tarefa; 256 MiB mantém alguns arquivos por partição, ou seja, paralelismo na leitura.
- `spark.ui.showConsoleProgress = false` — tira as barras de progresso do output das células, que
  em notebook só poluem (o progresso real está na Spark UI, em `localhost:4040`).

Observação: `driver.memory` só tem efeito se for definido **antes** da JVM subir — se a sessão já
existir no kernel, reinicie o kernel.

In [ ]:
DATASETS = next(
    path / "datasets" for path in Path.cwd().parents if (path / "datasets").is_dir()
)
SOURCE = DATASETS / "stocks_b3"
TARGET = DATASETS / "stocks_b3_delta"
FEATURES = DATASETS / "stocks_b3_features"

spark = configure_spark_with_delta_pip(
    SparkSession.builder.appName("stocks_b3")
    .master("local[4]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.driver.memory", "6g")
    .config("spark.sql.parquet.compression.codec", "zstd")
    .config("spark.databricks.delta.optimizeWrite.enabled", "true")
    .config("spark.databricks.delta.autoCompact.enabled", "true")
    .config("spark.databricks.delta.optimize.maxFileSize", "268435456")
    .config("spark.ui.showConsoleProgress", "false")
).getOrCreate()

### Passo 3 — Ler o parquet bruto

`spark.read.parquet(str(SOURCE))` lê todos os anos de uma vez. Confira:

- `printSchema()` — datas devem vir como `date`, preços como `double`, contagens como `long` (o
  gerador já tipou tudo).
- Contagem de linhas por ano: `groupBy(F.year("trade_date"))`, `count()`, ordenado por ano.

Um ano recente tem ~3 M de linhas porque **opções são ~87 % do arquivo**; ações à vista são ~8 %.
Guarde esse número — é o motivo do filtro do passo 5.

In [ ]:
df = spark.read.parquet(str(SOURCE))

df.printSchema()

df.count()

df.groupBy(F.year("trade_date").alias("year")).count().orderBy("year").show(10)

### Passo 4 — Criar o dataset otimizado (Delta particionado por ano)

Grave o DataFrame bruto em `TARGET` no formato Delta, particionado por uma coluna `year` derivada
de `trade_date` (`withColumn("year", F.year(...))`, depois `.write.format("delta")
.partitionBy("year").mode("overwrite").save(...)`).

Antes de gravar, cheque `DeltaTable.isDeltaTable(spark, str(TARGET))` e pule a escrita se já
existir — assim re-executar o notebook não refaz o trabalho. Para reprocessar (por exemplo após
baixar anos novos), apague o diretório `TARGET`.

Por que Delta em vez do parquet solto:

- **Partição por `year`** — as consultas do bloco 2 e do treino recortam por período; o Spark abre
  só os diretórios do intervalo.
- **Estatísticas por arquivo** (min/max de `ticker` e `trade_date`) — filtros por ativo ou data
  pulam arquivos inteiros (*data skipping*).
- **Escrita atômica** — se falhar no meio, não sobra diretório pela metade.

Termine lendo a tabela de volta (`spark.read.format("delta").load(...)`) por cima do mesmo `df` e
conferindo o `count()` contra o total do passo 3. Daqui em diante `df` é sempre a tabela Delta —
cada passo do bloco 2 reatribui `df = df.<transformação>`.

In [ ]:
if not DeltaTable.isDeltaTable(spark, str(TARGET)):
    (
        df.withColumn("year", F.year("trade_date"))
        .write.format("delta")
        .partitionBy("year")
        .mode("overwrite")
        .save(str(TARGET))
    )

df = spark.read.format("delta").load(str(TARGET))

df.count()

df.groupBy("year").count().orderBy("year").show(10)

## Bloco 2 — Tratamento e features

Todas as features são calculadas **por ativo, na ordem do tempo**. Defina uma vez a janela base
`by_ticker = Window.partitionBy("ticker").orderBy("trade_date")` e derive as variantes com
`.rowsBetween(-N + 1, 0)` — os últimos N pregões **incluindo o atual**. "Dias" aqui significa
pregões (linhas), não dias-calendário.

Cuidado com divisão por zero: o Spark 4 roda em modo ANSI por padrão e `x / 0` **lança erro** em vez
de devolver nulo. Onde o denominador pode ser zero, use `F.try_divide(numerador, denominador)`.

### Passo 5 — Filtrar somente ações

O COTAHIST mistura tudo que negocia na B3. Dois campos classificam cada linha:

| `market_type` | Mercado |
|---|---|
| `010` | **À vista** (ações, FIIs, ETFs, BDRs, units) |
| `020` | Fracionário (lote de 1, ticker termina em `F`) |
| `030` | Termo |
| `070` / `080` | Opções de compra / venda |
| `012` / `013` | Opções com exercício |

Dentro do mercado à vista, o `bdi_code` separa o tipo de papel:

| `bdi_code` | Papel |
|---|---|
| `02` | **Lote padrão** — ações negociadas normalmente |
| `12` | Fundos imobiliários |
| `14` | ETFs, BDRs e certificados |
| `10` | Direitos e recibos de subscrição |
| `34`–`42` | Leilão, recuperação judicial e outras situações especiais |

Ação "normal" = `market_type == "010"` **e** `bdi_code == "02"`. (Para incluir FIIs e ETFs seria
`bdi_code IN (02, 12, 14)`.)

Depois do `filter`, faça um `select` só com o que interessa daqui em diante: `trade_date`, `year`,
`ticker`, `company_name`, `specification`, `isin_code`, os quatro preços, `total_trades`,
`total_quantity`, `total_volume` e `quote_factor`. Os campos de opção e termo (`strike_price`,
`expiration_date`, `forward_term`, ...) ficam de fora.

Confira `count()` e `select("ticker").distinct().count()` — deve dar algumas centenas de tickers
por ano, não milhares.

**Entra:** `market_type`, `bdi_code`. **Sai:** `df` só com ações e com as colunas do `select`.

In [ ]:
df = df.filter(
    (F.col("market_type") == "010")
    & (F.col("bdi_code") == "02")
    & (~F.col("specification").contains("DRN"))
).select(
    "trade_date",
    "year",
    "ticker",
    "company_name",
    "specification",
    "isin_code",
    "open_price",
    "high_price",
    "low_price",
    "close_price",
    "total_trades",
    "total_quantity",
    "total_volume",
    "quote_factor",
)

df.count()

df.select("ticker").distinct().count()

### Passo 6 — Normalizar o preço pelo fator de cotação

Até 1997 a B3 cotava a maioria das ações **por lote de mil** (`quote_factor = 1000`): PETR4 aparece
a R$ 470 em 2000 quando valia R$ 0,47 por ação. Divida `open_price`, `high_price`, `low_price` e
`close_price` por `quote_factor` usando `F.try_divide` — em modo ANSI a divisão comum quebraria a
célula num fator zerado de linha antiga, e aqui nulo é melhor que exceção.

`withColumns` com um dict comprehension resolve os quatro de uma vez. Repare no papel duplo do
nome dentro da comprehension: como **chave** ele diz qual coluna gravar, como argumento do
`try_divide` ele diz qual coluna ler. Chave igual a uma coluna existente **substitui** aquela
coluna — o preço bruto não fica guardado, e por isso `quote_factor` já pode sair no `drop` logo em
seguida. Se algum dia quiser manter o bruto ao lado, é só a chave virar um nome novo
(`f"unit_{name}"`), e aí o fator tem de ficar para a linha continuar auditável.

Crie também `day_trade = close_price − open_price` (`F.try_subtract`), a variação intradiária em R$.
Como ela nasce na mesma expressão encadeada, depois das divisões, já sai em preço por ação. Duas
ressalvas para anotar:

- Em R$ a coluna **não é comparável entre ativos**: R$ 0,10 numa ação de R$ 1 é 10 %, na de R$ 100 é
  0,1 %. Se ela for para o modelo, a forma comparável é a razão (`close_price / open_price − 1`) ou
  o log (`ln(close_price) − ln(open_price)`).
- `try_subtract` existe para overflow de tipo inteiro; em `double` ele faz o mesmo que o `-` comum.
  Não custa nada, mas não está protegendo nada aqui.

Confira com `filter(ticker == "PETR4").orderBy("trade_date").show(5)` — nos anos 90 o preço deve
ficar na casa dos centavos, não das centenas.

**Atenção:** o COTAHIST **não** ajusta preços por desdobramento, grupamento ou provento. Num split
1:4 o preço cai 75 % de um dia para o outro e as features de janela (retorno, média móvel,
estocástico) registram isso como queda real. Para um modelo sério é preciso reconstruir a série
ajustada com os eventos corporativos — fica fora deste notebook, mas anote a limitação.

**Entra:** os quatro preços, `quote_factor`. **Sai:** os quatro preços por ação (mesmos nomes) e
`day_trade`.

In [ ]:
df = df.withColumns(
    {
        name: F.try_divide(name, "quote_factor")
        for name in ("open_price", "high_price", "low_price", "close_price")
    }
).drop("quote_factor")

df = df.withColumn("day_trade", F.col("close_price") - F.col("open_price"))

df.filter(F.col("ticker") == "PETR4").orderBy("trade_date").show(5)

### Passo 7 — Colunas em log e retorno logarítmico

Saem daqui duas coisas de naturezas diferentes.

**1. As cinco colunas em log** — `log_open_price`, `log_high_price`, `log_low_price`,
`log_close_price` e `log_day_trade`, num `withColumns` com dict comprehension de chave
`f"log_{name}"` (chave nova, coluna nova; o valor bruto continua ao lado). Em escala log,
variações percentuais iguais têm a mesma distância — 10 → 20 é o mesmo passo que 100 → 200 —, o que
achata a diferença entre uma ação de R$ 1 e uma de R$ 100 na entrada do modelo.

`F.log1p(x) = ln(1 + x)`. O `+1` é o que torna a função utilizável em `day_trade`, que zera todo dia
em que abertura e fechamento coincidem (`ln(0)` seria `-inf`). O preço da conveniência:

- **`log1p` não é `log`.** As colunas de preço ficam `ln(1 + p)`, deslocadas. Como feature de nível
  isso é inofensivo — o modelo aprende a escala que você der. O que **não** dá é tirar retorno
  subtraindo duas delas: `ln(1 + p_t) − ln(1 + p_{t−1})` não é o retorno. Com 9,31 → 9,50 essa
  subtração dá 1,83 % onde o retorno real é 2,02 %, e o erro cresce quanto menor o preço — pior
  justamente nas ações de centavos, que são as mais voláteis.
- **`log_day_trade` é nulo em queda relevante.** `log1p` fora do domínio (`x ≤ −1`) devolve NULL sem
  aviso: uma ação de R$ 50 que recua R$ 2 já cai nisso. Rode um `count` de nulos nessa coluna antes
  de levá-la ao modelo, e lembre que o sinal da variação se perde de qualquer forma (queda de
  R$ 0,30 e alta de R$ 0,30 viram valores de módulos diferentes, não simétricos).

**2. `log_return`** — o retorno contínuo de um pregão, aditivo no tempo (a soma dos diários é o
retorno do período). É a entrada do passo 8 e do OBV (*On-Balance Volume*) no passo 12, e por isso
precisa ser o log de verdade. Calcule direto do preço, sem passar pelas colunas acima:

`F.log(F.try_divide("close_price", F.lag("close_price").over(by_ticker)))`

`ln(p_t / p_{t−1})` é exatamente `ln(p_t) − ln(p_{t−1})`, sem o `1 +` no meio. `try_divide` cobre os
dois casos ruins do denominador — a primeira linha de cada ativo, onde o `lag` devolve nulo, e um
fechamento zerado — e `F.log` de zero ou negativo também devolve nulo, então a coluna nasce
limpa: nulo onde não há retorno definido, número no resto.

É aqui que `by_ticker = Window.partitionBy("ticker").orderBy("trade_date")` aparece pela primeira
vez, então **defina a janela nesta célula**; os passos 8 a 12 reaproveitam. O `partitionBy` é o que
faz o `lag` olhar a linha anterior **do mesmo ticker** e não o último papel da ordenação global.

**Entra:** os quatro preços, `day_trade`. **Sai:** `log_open_price`, `log_high_price`,
`log_low_price`, `log_close_price`, `log_day_trade`, `log_return`.

In [ ]:
by_ticker = Window.partitionBy("ticker").orderBy("trade_date")

df = df.withColumns(
    {
        f"log_{name}": F.log1p(name)
        for name in (
            "open_price",
            "high_price",
            "low_price",
            "close_price",
            "day_trade",
        )
    }
).withColumn(
    "log_return",
    F.log(F.try_divide("close_price", F.lag("close_price").over(by_ticker))),
)

df.filter(F.col("ticker") == "PETR4").orderBy("trade_date").select(
    "trade_date", "close_price", "log_close_price", "log_return"
).show(5)

### Passo 8 — Z-score modificado do retorno (janela de 90 pregões)

Depende de `log_return` e de `by_ticker`, ambos do passo 7.

O z-score clássico usa média e desvio padrão, que são puxados justamente pelos outliers que se quer
detectar: um tombo de 20 % infla o desvio padrão e passa a parecer normal dentro da própria
distribuição que ele distorceu. O **z-score modificado** (Iglewicz & Hoaglin) troca os dois por
mediana e MAD (*Median Absolute Deviation*, desvio absoluto mediano), que não se mexem quando a
minoria dos pontos vai para longe:

$$M_t = 0{,}6745 \cdot \frac{x_t - \mathrm{mediana}(x)}{\mathrm{MAD}(x)}, \qquad
\mathrm{MAD}(x) = \mathrm{mediana}\big(|x_i - \mathrm{mediana}(x)|\big)$$

- $x$ = `log_return` dos últimos 90 pregões do ativo, incluindo o atual.
- $0{,}6745$ é o quantil 75 % da normal padrão; com ele o MAD fica na mesma escala do desvio padrão
  para dados normais, e $|M| > 3{,}5$ é o corte usual de outlier.

O Spark não tem mediana exata em janela, então ela é montada na mão sobre um array:

1. **A janela.** `last_90 = by_ticker.rowsBetween(-89, 0)`. `by_ticker` já diz *de quem*
   (`partitionBy`) e *em que ordem* (`orderBy`); `rowsBetween` acrescenta a terceira peça, o
   *frame*: quais linhas em volta da atual entram na conta. A contagem é em **linhas** — `-89` = 89
   pregões atrás, `0` = a linha atual, 90 no total. O primo `rangeBetween` contaria pelo *valor* da
   coluna de ordenação (dias de calendário, com fim de semana e feriado dentro), que não é o que se
   quer. O frame é cortado na borda da partição, então as primeiras linhas de cada ativo usam o que
   houver, sem tratamento especial. Tudo isso é só especificação: nada é calculado até um `.over()`.
2. **O array.** `return_window = F.collect_list("log_return").over(last_90)` — até 90 números por
   linha. `collect_list` **descarta nulos**, e é isso que faz o array sair **vazio** na primeira
   linha de cada ativo, onde `log_return` é nulo.
3. **`array_median(values)`.** `F.array_sort` ordena, `F.size` dá o tamanho, e os elementos do meio
   saem em `F.floor((size + 1) / 2)` e `F.ceil((size + 1) / 2)` com `.cast("int")` — em tamanho
   ímpar os dois índices coincidem no elemento central, em tamanho par são os dois vizinhos e a
   média deles é a mediana. Envolva o resultado em `F.when(size > 0, ...)`: **isso não é opcional**.
   Com array vazio o índice calculado é 0 e `F.try_element_at` **lança**
   `[INVALID_INDEX_OF_ZERO]` — o `try_` cobre índice fora do intervalo, não o índice zero, que é
   inválido por definição (array em SQL começa em 1). O `CASE WHEN` só avalia o ramo cuja condição
   casa, e é exatamente isso que impede a chamada de acontecer.
4. **A mediana.** `return_median = array_median("return_window")`.
5. **O MAD.** `return_mad = array_median(F.transform("return_window", lambda x: F.abs(x -
   F.col("return_median"))))`. O lambda de uma função de ordem superior pode referenciar colunas de
   fora do array — é assim que a mediana da linha entra no cálculo do desvio de cada elemento.
6. **O z-score.** `0.6745 * (log_return − return_median) / return_mad`, dentro de
   `F.when(return_mad > 0, ...)`.
7. **Limpeza.** `.drop("return_window")` — guardar 90 números por linha até o fim do pipeline não
   faz sentido.

Quando o resultado sai nulo, e é mais frequente do que parece:

- Primeira linha do ativo: `log_return` nulo → array vazio → mediana nula.
- `return_mad == 0`: acontece sempre que **mais da metade** dos 90 retornos é igual à mediana. Papel
  parado, que repete o mesmo fechamento dia após dia, tem mediana 0 e MAD 0. Não é falha do
  cálculo — é o MAD dizendo que não existe dispersão para normalizar. É também o motivo de o corte
  $|M| > 3{,}5$ só significar alguma coisa em papel com liquidez.

**Entra:** `log_return`. **Sai:** `return_median`, `return_mad`, `return_modified_zscore`.

In [ ]:
last_90 = by_ticker.rowsBetween(-89, 0)


def array_median(values):
    ordered = F.array_sort(values)
    size = F.size(ordered)
    lower = F.try_element_at(ordered, F.floor((size + 1) / 2).cast("int"))
    upper = F.try_element_at(ordered, F.ceil((size + 1) / 2).cast("int"))
    return F.when(size > 0, (lower + upper) / 2)


df = df.withColumn("return_window", F.collect_list("log_return").over(last_90))

df = df.withColumn("return_median", array_median("return_window"))

df = df.withColumn(
    "return_mad",
    array_median(
        F.transform("return_window", lambda x: F.abs(x - F.col("return_median")))
    ),
)

df = df.withColumn(
    "return_modified_zscore",
    F.when(
        F.col("return_mad") > 0,
        0.6745 * (F.col("log_return") - F.col("return_median")) / F.col("return_mad"),
    ),
).drop("return_window")

df.filter(F.col("ticker") == "PETR4").orderBy(F.desc("trade_date")).select(
    "trade_date", "log_return", "return_median", "return_mad", "return_modified_zscore"
).show(10)

df.show(10)

### Passo 9 — Estocástico lento (14, 3, 3)

O oscilador estocástico mede **onde o fechamento de hoje está dentro da faixa de preços dos últimos
N pregões**, de 0 a 100. São três etapas encadeadas:

1. **%K rápido** — posição do fechamento na faixa dos últimos 14 pregões:
   $$\%K_{rápido} = 100 \cdot \frac{close_t - \min(low_{t-13..t})}{\max(high_{t-13..t}) - \min(low_{t-13..t})}$$
   Fechou na máxima dos 14 dias → 100; na mínima → 0.
2. **%K lento** — média simples de 3 pregões do %K rápido. É o suavizador que dá o nome "lento":
   tira o ruído dia a dia.
3. **%D lento** — média simples de 3 pregões do %K lento. É a "linha de sinal"; o cruzamento de %K
   com %D é o gatilho clássico.

Leitura usual: acima de 80 = sobrecomprado, abaixo de 20 = sobrevendido.

Implementação: janelas `last_14 = by_ticker.rowsBetween(-13, 0)` e `last_3 = by_ticker
.rowsBetween(-2, 0)`; `F.min("low_price")` e `F.max("high_price")` sobre `last_14`; `F.avg` sobre
`last_3` para os dois suavizamentos. Se máxima e mínima da janela coincidem (papel sem negociação
real por 14 dias) o denominador é zero — use `F.try_divide`.

**Entra:** `high_price`, `low_price`, `close_price`. **Sai:** `fast_k`, `slow_k`, `slow_d`.

### Passo 10 — Média móvel de 90 pregões

- `sma_90 = F.avg("close_price").over(last_90)` — reaproveite a janela do passo 8. 90 pregões ≈ 4
  meses e meio de calendário.
- `sma_90_distance = close_price / sma_90 − 1` — quanto o preço está acima (+) ou abaixo (−) da
  média, em fração. É essa que entra no modelo, porque é comparável entre ativos de preços
  diferentes; a média em R$ sozinha não é.

**Entra:** `close_price`. **Sai:** `sma_90`, `sma_90_distance`.

### Passo 11 — Array dos últimos 6 meses de alta (1) ou baixa (0)

Precisa mudar de granularidade (diário → mensal) e voltar:

1. Crie `month = F.trunc("trade_date", "month")` e agrupe por (`ticker`, `month`) pegando o **último
   fechamento do mês**: `F.max_by("close_price", "trade_date")`. Chame de `month_close`.
2. Janela mensal `by_ticker_month = Window.partitionBy("ticker").orderBy("month")`.
   `month_up = 1` se `month_close` superou o do mês anterior (`F.lag`), senão `0` — compare e faça
   `.cast("int")`.
3. `last_6_months_up = F.collect_list("month_up").over(by_ticker_month.rowsBetween(-6, -1))`.
   O `-1` exclui o mês corrente **de propósito**: no dia 10 o mês ainda não fechou, e usar o
   resultado dele seria vazar o futuro para dentro da feature.
4. Volte ao diário: adicione a mesma coluna `month` em `df` e faça `join` com o mensal por
   (`ticker`, `month`); depois descarte `month`. Todo pregão de agosto recebe o array de fevereiro a
   julho. O agregado mensal é a única variável intermediária que precisa de outro nome (por exemplo
   `monthly`) — `df` continua sendo o diário.

Detalhes que valem anotar: o array tem menos de 6 posições nos primeiros meses do ativo, e "mês
anterior" é o mês anterior **com negociação** — papel que ficou meses parado pula o intervalo.

**Entra:** `trade_date`, `close_price`. **Sai:** `last_6_months_up` (`array<int>`).

### Passo 12 — Indicadores de volume

`total_volume` é o financeiro do dia em R$; `total_quantity` é a quantidade de ações negociadas.
Janela `last_20 = by_ticker.rowsBetween(-19, 0)` (≈ 1 mês).

- `log_volume = ln(1 + total_volume)` — `F.log1p`. Volume tem cauda pesada (dia de evento negocia
  50× a média); em log a distribuição fica utilizável, e o `+1` evita `ln(0)` em dia sem negócio.
- `volume_sma_20 = F.avg("total_volume").over(last_20)`.
- `volume_ratio = total_volume / volume_sma_20` — 2,0 significa "hoje negociou o dobro do normal".
  É a leitura mais direta de volume anômalo. `try_divide`.
- `volume_zscore_20 = (total_volume − volume_sma_20) / F.stddev("total_volume").over(last_20)` —
  z-score clássico contra a mesma janela; complementa o `ratio` levando em conta a dispersão do
  ativo. `try_divide`.
- `obv` (*On-Balance Volume*) — soma acumulada de `total_quantity` com sinal: `+` nos dias em que
  `log_return > 0`, `−` quando `< 0`, `0` quando estável (`F.when(...).when(...).otherwise(0)`).
  Depois `F.sum(...).over(by_ticker)` — sem `rowsBetween`, o frame padrão de uma janela ordenada é
  "do início até a linha atual", que é exatamente o acumulado. Mede se o volume está entrando ou
  saindo do papel ao longo do tempo.

**Entra:** `total_volume`, `total_quantity`, `log_return`. **Sai:** `log_volume`, `volume_sma_20`,
`volume_ratio`, `volume_zscore_20`, `obv`.

### Passo 13 — Resultado e gravação

1. `printSchema()` — confira que todas as colunas dos passos 7–12 estão lá com o tipo esperado
   (`last_6_months_up` deve ser `array<int>`).
2. Olhe PETR4 nos últimos 10 pregões (`orderBy(F.desc("trade_date"))`) com um `select` só das
   features, `show(10, truncate=False)`. Sanidade: `slow_k` e `slow_d` entre 0 e 100,
   `return_modified_zscore` quase sempre entre −3,5 e 3,5, `volume_ratio` em torno de 1.
3. Grave em `FEATURES` como Delta particionado por `year`, `mode("overwrite")` — as features são
   recalculadas do zero a cada execução. Essa tabela é a entrada do próximo bloco (modelo).